# OpenEnv Colab Submission Notebook

This notebook runs the full submission pipeline for the multi-app enterprise orchestration environment:
1. Validate OpenEnv compatibility
2. Run world-modeling demo
3. Train environment-grounded policy
4. Evaluate baseline/mid/trained reward progression
5. Print artifact summary for README/blog/video evidence


In [1]:
import json
import os
import subprocess
from pathlib import Path

ROOT = Path.cwd()
print(f"Working directory: {ROOT}")
print("Python:", os.sys.version.split()[0])


Working directory: /workspaces/scaler-final
Python: 3.12.1


## (Optional for Colab) Install dependencies

Uncomment this cell when running in a clean Colab runtime.

In [2]:
# !pip install -q -r requirements.txt
# !pip install -q -e .


In [3]:
def run(cmd: str):
    print(f"\n$ {cmd}")
    completed = subprocess.run(cmd, shell=True, check=True, text=True, capture_output=True)
    if completed.stdout.strip():
        print(completed.stdout)
    if completed.stderr.strip():
        print(completed.stderr)
    return completed


## 1) Validate environment

In [4]:
run('openenv validate')


$ openenv validate


[OK] scaler-final: Ready for multi-mode deployment



CompletedProcess(args='openenv validate', returncode=0, stdout='[OK] scaler-final: Ready for multi-mode deployment\n', stderr='')

## 2) World-modeling behavior demo

In [5]:
run('python world_modeling_demo.py')


$ python world_modeling_demo.py


WORLD MODELING VALIDATION DEMO
Hidden world state: multi-app enterprise dataset (CRM + Billing + Support)
Visible observation: schema, missing/duplicate counts, KPIs, actor messages
Agent actions: analyze -> delegate/reconcile -> validate -> report

TASK task_missing_values
reset: dataset=crm_contacts shape=(132, 7) missing=108 duplicates=24 actions=0 drift=False kpi={'quality_index': 0.864935, 'backlog_pressure': 1.0, 'policy_compliance': 0.5, 'workflow_latency': 1.0}
step=1 action=analyze reward=0.225 missing 108->108 duplicates 24->24 drift=False grade=0.202
step=2 action=impute reward=0.617 missing 108->0 duplicates 24->22 drift=False grade=0.990
step=3 action=deduplicate reward=0.700 missing 0->0 duplicates 22->0 drift=False grade=0.990
step=4 action=validate reward=0.250 missing 0->0 duplicates 0->0 drift=False grade=0.990
step=5 action=report_findings reward=0.500 missing 0->0 duplicates 0->0 drift=False grade=0.990
step=6 action=report_findings reward=0.500 missing 0->0 duplica

CompletedProcess(args='python world_modeling_demo.py', returncode=0, stdout="WORLD MODELING VALIDATION DEMO\nHidden world state: multi-app enterprise dataset (CRM + Billing + Support)\nVisible observation: schema, missing/duplicate counts, KPIs, actor messages\nAgent actions: analyze -> delegate/reconcile -> validate -> report\n\nTASK task_missing_values\nreset: dataset=crm_contacts shape=(132, 7) missing=108 duplicates=24 actions=0 drift=False kpi={'quality_index': 0.864935, 'backlog_pressure': 1.0, 'policy_compliance': 0.5, 'workflow_latency': 1.0}\nstep=1 action=analyze reward=0.225 missing 108->108 duplicates 24->24 drift=False grade=0.202\nstep=2 action=impute reward=0.617 missing 108->0 duplicates 24->22 drift=False grade=0.990\nstep=3 action=deduplicate reward=0.700 missing 0->0 duplicates 22->0 drift=False grade=0.990\nstep=4 action=validate reward=0.250 missing 0->0 duplicates 0->0 drift=False grade=0.990\nstep=5 action=report_findings reward=0.500 missing 0->0 duplicates 0->0

## 3) Train policy from environment rollouts

In [6]:
run('python training/trl_sft_training.py')


$ python training/trl_sft_training.py


CompletedProcess(args='python training/trl_sft_training.py', returncode=0, stdout='', stderr='')

## 4) Evaluate reward improvement (baseline/mid/trained)

In [7]:
run('python training/evaluate_reward_improvement.py')


$ python training/evaluate_reward_improvement.py


CompletedProcess(args='python training/evaluate_reward_improvement.py', returncode=0, stdout='', stderr='')

## 5) Print key evidence artifacts

In [8]:
artifacts = Path('artifacts')
for fp in [
    artifacts / 'reward_progression.csv',
    artifacts / 'trl_sft_training_metrics.json',
    artifacts / 'training_curve.csv',
]:
    print(f"\n=== {fp} ===")
    print(fp.read_text(encoding='utf-8')[:4000])



=== artifacts/reward_progression.csv ===
stage,average_score,std_score
baseline,0.442923,0.097888
mid,0.636090,0.203447
trained,0.726893,0.160171


=== artifacts/trl_sft_training_metrics.json ===
{
  "training_mode": "environment_grounded_policy_search",
  "optimizer": "hill_climbing_over_action_programs",
  "episodes_per_task_baseline": 6,
  "episodes_per_task_eval": 10,
  "iterations": 60,
  "accepted_updates": 18,
  "baseline_average_score": 0.476924,
  "trained_average_score": 0.72203,
  "improvement": 0.245106,
  "baseline_std": 0.233253,
  "trained_std": 0.156272,
  "task_scores_baseline": {
    "task_missing_values": 0.700224,
    "task_duplicate_handling": 0.09359,
    "task_complex_validation": 0.612486,
    "task_enterprise_orchestration": 0.501395
  },
  "task_scores_trained": {
    "task_missing_values": 0.885896,
    "task_duplicate_handling": 0.513478,
    "task_complex_validation": 0.858523,
    "task_enterprise_orchestration": 0.630221
  },
  "task_improvement": {
    

## 6) Quick inference sanity check

In [9]:
run('INFERENCE_BACKEND=local INFERENCE_SEED=2026 python inference.py')


$ INFERENCE_BACKEND=local INFERENCE_SEED=2026 python inference.py


[START] task=task_missing_values seed=2026 backend=local model=local_policy
[STEP] task=task_missing_values step=1 action=analyze reward=0.225000 done=false
[STEP] task=task_missing_values step=2 action=impute reward=0.616667 done=false
[STEP] task=task_missing_values step=3 action=deduplicate reward=0.700000 done=false
[STEP] task=task_missing_values step=4 action=validate reward=0.250000 done=false
[STEP] task=task_missing_values step=5 action=report_findings reward=0.500000 done=false
[STEP] task=task_missing_values step=6 action=report_findings reward=0.500000 done=true
[END] task=task_missing_values score=0.990000 steps=6
[START] task=task_duplicate_handling seed=2027 backend=local model=local_policy
[STEP] task=task_duplicate_handling step=1 action=analyze reward=0.212500 done=false
[STEP] task=task_duplicate_handling step=2 action=deduplicate reward=0.726316 done=false
[STEP] task=task_duplicate_handling step=3 action=validate reward=0.250000 done=false
[STEP] task=task_duplicat

CompletedProcess(args='INFERENCE_BACKEND=local INFERENCE_SEED=2026 python inference.py', returncode=0, stdout='[START] task=task_missing_values seed=2026 backend=local model=local_policy\n[STEP] task=task_missing_values step=1 action=analyze reward=0.225000 done=false\n[STEP] task=task_missing_values step=2 action=impute reward=0.616667 done=false\n[STEP] task=task_missing_values step=3 action=deduplicate reward=0.700000 done=false\n[STEP] task=task_missing_values step=4 action=validate reward=0.250000 done=false\n[STEP] task=task_missing_values step=5 action=report_findings reward=0.500000 done=false\n[STEP] task=task_missing_values step=6 action=report_findings reward=0.500000 done=true\n[END] task=task_missing_values score=0.990000 steps=6\n[START] task=task_duplicate_handling seed=2027 backend=local model=local_policy\n[STEP] task=task_duplicate_handling step=1 action=analyze reward=0.212500 done=false\n[STEP] task=task_duplicate_handling step=2 action=deduplicate reward=0.726316 d